## Imports and Setup:

In [ ]:
# Mount Google Drive for model saving
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/GNN_MEA/victim_models/'
import os
os.makedirs(SAVE_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.1 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.model_selection import StratifiedKFold
import numpy as np

## Setting up Datasets (importing + test/train/split) and GCN Victim Model Architecture:

In [ ]:
# Setting up Datasets
DATASET_NAMES = [
    'AIDS', 'MUTAG', 'PTC_FM', 'NCI1', 'Tox21_AhR_training',
]

datasets = {}
for name in DATASET_NAMES:
    ds = TUDataset(root=f'data/{name}', name=name)

    datasets[name] = ds

    assert ds[0].x is not None, f"{name}: features still None!"
    print(f"  Verified: feature dim = {ds[0].x.shape[1]}")

Processing...
Done!


  Verified: feature dim = 38


Processing...
Done!


  Verified: feature dim = 7


Processing...
Done!


...

Processing...
Done!


  Verified: feature dim = 37


Processing...


  Verified: feature dim = 50


Done!


In [ ]:
# Train/test/split: 60% victim train, 20% shadow, 20% test (stratified)
from sklearn.model_selection import train_test_split

def split_dataset(dataset, seed=42): # Random seed is set to 42 for reproducibility
  labels = [data.y.item() for data in dataset]

  # First split: 60% train, 40% remaining
  train_idx, remaining_idx = train_test_split(
    range(len(dataset)), test_size=0.4,
    stratify=labels, random_state=seed
  )

  # Second split: 50/50 on remaining = 20% shadow, 20% test
  remaining_labels = [labels[i] for i in remaining_idx]
  shadow_idx, test_idx = train_test_split(
    remaining_idx, test_size=0.5,
    stratify=remaining_labels, random_state=seed
  )

  return train_idx, shadow_idx, test_idx

In [ ]:
# GCN Victim Model Architecture
class GCN(nn.Module):
  def __init__(self, in_dim, hidden_dim, num_classes, dropout=0.5): # Can play around with the dropout value too if needed
    super().__init__()
    # 3-layer GCN with final classification layer (can experiment with 64 or 128 hidden_dim)
    self.conv1 = GCNConv(in_dim, hidden_dim)
    self.conv2 = GCNConv(hidden_dim, hidden_dim)
    self.conv3 = GCNConv(hidden_dim, hidden_dim)
    self.classifier = nn.Linear(hidden_dim, num_classes)
    self.dropout = dropout

  def forward(self, x, edge_index, batch):
    # Message Passing + Dropout (randomly zeros out neurons)
    x = F.relu(self.conv1(x, edge_index))
    x = F.dropout(x, p=self.dropout, training=self.training)
    x = F.relu(self.conv2(x, edge_index))
    x = F.dropout(x, p=self.dropout, training=self.training)
    x = F.relu(self.conv3(x, edge_index))
    # Pools node vectors into graph vector
    x = global_mean_pool(x, batch)
    # Classifies
    x = self.classifier(x)
    return x

In [ ]:
def get_feature_dim(dataset):
    """Get actual feature dimension (handles manual feature assignment)."""
    return dataset[0].x.shape[1]

In [ ]:
# After training, check prediction diversity
def check_predictions(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            preds.extend(out.argmax(dim=1).cpu().tolist())
    unique, counts = np.unique(preds, return_counts=True)
    print(f"  Prediction distribution: {dict(zip(unique, counts))}")
    if len(unique) == 1:
        print("  WARNING: Model predicts single class!")
    return len(unique) > 1

## Training or Loading the GCN Victim Model:

In [ ]:
# Loading the saved victim model function
def load_victim(name, dataset, device='cuda'):
    save_path = os.path.join(SAVE_DIR, f'{name}_victim.pt')
    checkpoint = torch.load(save_path, map_location=device, weights_only=False)

    model = GCN(
        checkpoint['in_dim'],
        checkpoint['config']['hidden_dim'],
        checkpoint['num_classes']
    ).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    return model, checkpoint

In [ ]:
# Training the victim model function
def train_victim(dataset, train_idx, test_idx, device='cuda'):
    train_set = [dataset[i] for i in train_idx]
    test_set = [dataset[i] for i in test_idx]

    # Fix float labels
    for data in train_set + test_set:
        data.y = data.y.long()

    in_dim = get_feature_dim(dataset)
    num_classes = dataset.num_classes

    # Hyperparameter search grid
    epoch_choices = [300, 500, 700, 1000]
    hidden_choices = [64, 128]

    best_acc = 0
    best_model = None
    best_config = {}

    for hidden_dim in hidden_choices:
        for max_epochs in epoch_choices:
            model = GCN(in_dim, hidden_dim, num_classes).to(device)
            optimizer = torch.optim.Adam(
                model.parameters(), lr=0.001, weight_decay=5e-4
            )
            loss_fn = nn.CrossEntropyLoss()
            train_loader = DataLoader(train_set, batch_size=32,
                                      shuffle=True)
            test_loader = DataLoader(test_set, batch_size=32)

            # Train
            model.train()
            for epoch in range(max_epochs):
                for batch in train_loader:
                    batch = batch.to(device)
                    pred = model(batch.x, batch.edge_index,
                                 batch.batch)
                    loss = loss_fn(pred, batch.y)
                    loss.backward()
                    optimizer.step()
                    optimizer.zero_grad()

            # Evaluate
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for batch in test_loader:
                    batch = batch.to(device)
                    pred = model(batch.x, batch.edge_index,
                                 batch.batch)
                    correct += (pred.argmax(1) == batch.y).sum().item()
                    total += batch.y.size(0)

            acc = correct / total
            print(f"  hidden={hidden_dim}, epochs={max_epochs}: "
                  f"acc={acc:.4f}")

            if acc > best_acc:
                best_acc = acc
                best_model = model
                best_config = {'hidden_dim': hidden_dim,
                               'epochs': max_epochs}

    # Check prediction diversity on test set
    test_loader = DataLoader(test_set, batch_size=32)
    is_valid = check_predictions(best_model, test_loader, device)

    print(f"  Best: {best_config}, acc={best_acc:.4f}, "
          f"valid={is_valid}")
    return best_model, best_config, best_acc

In [ ]:
# Run Training Across All Datasets
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Store results
victim_models = {}
victim_configs = {}
dataset_splits = {}

RUN_DATASETS = [
    'AIDS', 'MUTAG', 'PTC_FM', 'NCI1', 'Tox21_AhR_training',
]

for name in RUN_DATASETS:
    print(f"\n{'='*50}")
    print(f"Training victim on {name}")
    print(f"{'='*50}")

    ds = datasets[name]
    save_path = os.path.join(SAVE_DIR, f'{name}_victim.pt')

    # Skip training if model already saved
    if os.path.exists(save_path):
        print(f"  Already trained, loading from {save_path}")
        model, checkpoint = load_victim(name, ds, device)
        victim_models[name] = model
        victim_configs[name] = checkpoint['config']
        # Load saved splits to ensure consistency
        dataset_splits[name] = {
            'train': checkpoint['train_idx'],
            'shadow': checkpoint['shadow_idx'],
            'test': checkpoint['test_idx'],
        }
        print(f"  Config: {checkpoint['config']}, "
              f"acc={checkpoint['accuracy']:.4f}")
        continue  # Skip to next dataset

    train_idx, shadow_idx, test_idx = split_dataset(ds)

    print(f"  Split: {len(train_idx)} train, {len(shadow_idx)} shadow, "
          f"{len(test_idx)} test")

    # Check label distribution
    train_labels = [ds[i].y.item() for i in train_idx]
    unique, counts = np.unique(train_labels, return_counts=True)
    print(f"  Train label dist: {dict(zip(unique, counts))}")

    model, config, acc = train_victim(ds, train_idx, test_idx, device)
    victim_models[name] = model
    victim_configs[name] = config

    # Store splits in memory + save to disk
    dataset_splits[name] = {
        'train': train_idx,
        'shadow': shadow_idx,
        'test': test_idx,
    }

    torch.save({
        'model_state_dict': model.state_dict(),
        'config': config,
        'accuracy': acc,
        'in_dim': get_feature_dim(ds),
        'num_classes': ds.num_classes,
        'train_idx': train_idx,
        'shadow_idx': shadow_idx,
        'test_idx': test_idx,
    }, save_path)
    print(f"  Saved to {save_path}")

print("\n\nSummary:")
print("-" * 50)
for name in RUN_DATASETS:
    print(f"{name}: {victim_configs[name]}")

Using device: cuda

Training victim on AIDS
  Already trained, loading from /content/drive/MyDrive/GNN_MEA/victim_models/AIDS_victim.pt
  Config: {'hidden_dim': 128, 'epochs': 1000}, acc=0.8700

Training victim on MUTAG
  Already trained, loading from /content/drive/MyDrive/GNN_MEA/victim_models/MUTAG_victim.pt
  Config: {'hidden_dim': 64, 'epochs': 300}, acc=0.7895

Training victim on PTC_FM
  Already trained, loading from /content/drive/MyDrive/GNN_MEA/victim_models/PTC_FM_victim.pt
  Config: {'hidden_dim': 128, 'epochs': 500}, acc=0.6571

Training victim on NCI1
  Already trained, loading from /content/drive/MyDrive/GNN_MEA/victim_models/NCI1_victim.pt
  Config: {'hidden_dim': 64, 'epochs': 1000}, acc=0.6727

Training victim on Tox21_AhR_training
  Already trained, loading from /content/drive/MyDrive/GNN_MEA/victim_models/Tox21_AhR_training_victim.pt
  Config: {'hidden_dim': 64, 'epochs': 700}, acc=0.8898


Summary:
--------------------------------------------------
AIDS: {'hidden_d

## Locating Decision Boundary (and Constructing Decision Boundary Pairs for each Datasets):

In [ ]:
# Boundary Detection on Shadow Dataset
from torch_geometric.utils import to_dense_adj, dense_to_sparse

def get_prediction(model, data, device='cuda'):
    """Get victim model prediction for a single graph."""
    model.eval()
    with torch.no_grad():
        data = data.clone().to(device)
        # Create a batch of size 1
        batch = torch.zeros(data.num_nodes, dtype=torch.long).to(device)
        pred = model(data.x, data.edge_index, batch)
        return pred.argmax(dim=1).item()

def flip_edge(edge_index, num_nodes, i, j):
    """Flip edge (i,j) in the graph: add if absent, remove if present."""
    # Convert to adjacency matrix
    A = to_dense_adj(edge_index, max_num_nodes=num_nodes)[0]
    # Flip: 0->1 or 1->0 (undirected so flip both)
    A[i, j] = 1 - A[i, j]
    A[j, i] = 1 - A[j, i]
    # Convert back to edge_index
    new_edge_index, _ = dense_to_sparse(A)
    return new_edge_index

In [ ]:
import time

def classify_boundary(model, dataset, shadow_idx, device='cuda'):
    """Find boundary pairs and non-boundary graphs.

    Boundary pair: (original graph, flipped graph) with different predictions.
    Non-boundary: graph where no single flip changes prediction.
    """
    boundary_pairs = []   # list of (orig_idx, flipped_data, orig_pred, flipped_pred)
    non_boundary = []      # list of idx
    start_time = time.time()

    for count, idx in enumerate(shadow_idx):
        data = dataset[idx]
        n = data.num_nodes
        orig_pred = get_prediction(model, data, device)

        found_pair = False

        for i in range(n):
            for j in range(i + 1, n):
                new_edge_index = flip_edge(
                    data.edge_index, n, i, j
                )
                flipped_data = data.clone()
                flipped_data.edge_index = new_edge_index

                new_pred = get_prediction(model, flipped_data, device)
                if new_pred != orig_pred:
                    # Store the pair: original + flipped with their labels
                    boundary_pairs.append({
                        'orig_idx': idx,
                        'flipped_data': flipped_data.cpu(),
                        'orig_pred': orig_pred,
                        'flipped_pred': new_pred,
                        'flip_edge': (i, j),
                    })
                    found_pair = True
                    break
            if found_pair:
                break

        if not found_pair:
            non_boundary.append(idx)

        if (count + 1) % 10 == 0:
            elapsed = time.time() - start_time
            per_graph = elapsed / (count + 1)
            remaining = per_graph * (len(shadow_idx) - count - 1)
            print(f"    [{count+1}/{len(shadow_idx)}] "
                  f"{len(boundary_pairs)} pairs, "
                  f"{len(non_boundary)} non-boundary | "
                  f"{per_graph:.1f}s/graph, "
                  f"~{remaining/60:.1f}min remaining")

    print(f"    Done: {len(boundary_pairs)} boundary pairs, "
          f"{len(non_boundary)} non-boundary")
    return boundary_pairs, non_boundary

In [ ]:
# Run Boundary Detection
# Directory on Drive to persist boundary detection results
BOUNDARY_DIR = '/content/drive/MyDrive/GNN_MEA/boundary_results/'
os.makedirs(BOUNDARY_DIR, exist_ok=True)

boundary_results = {}

for name in RUN_DATASETS:

    print(f"\n{'='*50}")
    print(f"Boundary detection: {name}")
    print(f"{'='*50}")

    ds = datasets[name]
    model = victim_models[name]
    shadow_idx = dataset_splits[name]['shadow']

    # Skip if boundary detection already done for this dataset
    save_path = os.path.join(BOUNDARY_DIR, f'{name}_boundary.pt')
    if os.path.exists(save_path):
        print(f"  Already computed, loading from {save_path}")
        boundary_results[name] = torch.load(save_path, weights_only=False)
        b = boundary_results[name]
        print(f"  {len(b['boundary_pairs'])} pairs, "
              f"{len(b['non_boundary'])} non-boundary")
        continue

    print(f"  Shadow set size: {len(shadow_idx)}")

    # Exhaustive single edge flip search on shadow dataset
    boundary_pairs, non_boundary = classify_boundary(
        model, ds, shadow_idx, device
    )

    # Store: boundary_pairs = list of {orig_idx, flipped_data, preds, edge}
    #        non_boundary = list of graph indices far from decision boundary
    boundary_results[name] = {
        'boundary_pairs': boundary_pairs,
        'non_boundary': non_boundary,
    }

    # Save immediately so progress isn't lost if later datasets crash
    torch.save(boundary_results[name], save_path)
    print(f"  Saved to {save_path}")


Boundary detection: AIDS
  Already computed, loading from /content/drive/MyDrive/GNN_MEA/boundary_results/AIDS_boundary.pt
  127 pairs, 273 non-boundary

Boundary detection: MUTAG
  Already computed, loading from /content/drive/MyDrive/GNN_MEA/boundary_results/MUTAG_boundary.pt
  11 pairs, 27 non-boundary

Boundary detection: PTC_FM
  Already computed, loading from /content/drive/MyDrive/GNN_MEA/boundary_results/PTC_FM_boundary.pt
  32 pairs, 38 non-boundary

Boundary detection: NCI1
  Already computed, loading from /content/drive/MyDrive/GNN_MEA/boundary_results/NCI1_boundary.pt
  338 pairs, 484 non-boundary

Boundary detection: Tox21_AhR_training
  Already computed, loading from /content/drive/MyDrive/GNN_MEA/boundary_results/Tox21_AhR_training_boundary.pt
  69 pairs, 1565 non-boundary


## Boundary vs Non-boundary vs Hybrid vs Random

In [ ]:
# Load boundary results from Drive
BOUNDARY_DIR = '/content/drive/MyDrive/GNN_MEA/boundary_results/'
boundary_results = {}
for name in ['AIDS','MUTAG', 'PTC_FM', 'NCI1', 'Tox21_AhR_training']:
    path = os.path.join(BOUNDARY_DIR, f'{name}_boundary.pt')
    boundary_results[name] = torch.load(path, weights_only=False)
    b = boundary_results[name]
    print(f"{name}: {len(b['boundary_pairs'])} pairs, "
          f"{len(b['non_boundary'])} non-boundary")

AIDS: 127 pairs, 273 non-boundary
MUTAG: 11 pairs, 27 non-boundary
PTC_FM: 32 pairs, 38 non-boundary
NCI1: 338 pairs, 484 non-boundary
Tox21_AhR_training: 69 pairs, 1565 non-boundary


In [ ]:
# Train Surrogate
def train_surrogate(train_data, in_dim, num_classes, device='cuda'):
    model = GCN(in_dim, hidden_dim=64, num_classes=num_classes,
                dropout=0.0).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001,
                                  weight_decay=0)
    loss_fn = nn.CrossEntropyLoss()

    for data in train_data:
        data.y = data.y.long()

    loader = DataLoader(train_data, batch_size=8, shuffle=True)

    model.train()
    for epoch in range(500):
        total_loss = 0
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch.x, batch.edge_index, batch.batch)
            loss = loss_fn(pred, batch.y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        if epoch in [0, 99, 199, 299, 399, 499]:
            print(f"      Epoch {epoch}: loss={total_loss:.4f}")

    return model

In [ ]:
# Evaluate Fidelity
def evaluate_fidelity(surrogate, victim, dataset, test_idx, device='cuda'):
    """Fidelity = agreement rate between surrogate and victim on test set."""
    surrogate.eval()
    victim.eval()
    agree = 0
    total = 0

    with torch.no_grad():
        for idx in test_idx:
            data = dataset[idx].clone().to(device)
            batch = torch.zeros(data.num_nodes, dtype=torch.long).to(device)

            surr_pred = surrogate(data.x, data.edge_index, batch).argmax(1).item()
            vic_pred = victim(data.x, data.edge_index, batch).argmax(1).item()

            if surr_pred == vic_pred:
                agree += 1
            total += 1

    fidelity = agree / total
    return fidelity

In [ ]:
def build_all_training_sets(dataset, boundary_pairs, non_boundary_idx,
                            shadow_idx, model, device='cuda'):
    """Build 4 equal-sized training sets: boundary, non-boundary, hybrid, random."""
    # --- Boundary data ---
    boundary_data = []
    for pair in boundary_pairs:
        orig = dataset[pair['orig_idx']].clone()
        orig.y = torch.tensor([pair['orig_pred']])
        boundary_data.append(orig)
        flipped = pair['flipped_data'].clone()
        flipped.y = torch.tensor([pair['flipped_pred']])
        boundary_data.append(flipped)

    # --- Non-boundary data (labeled by victim) ---
    nb_all = []
    for idx in non_boundary_idx:
        data = dataset[idx].clone()
        pred = get_prediction(model, data, device)
        data.y = torch.tensor([pred])
        nb_all.append(data)

    # --- All shadow samples labeled by victim ---
    all_shadow = []
    for idx in shadow_idx:
        data = dataset[idx].clone()
        pred = get_prediction(model, data, device)
        data.y = torch.tensor([pred])
        all_shadow.append(data)

    # --- Target size: minimum across what each method can provide ---
    n_boundary = len(boundary_data)
    n_nb = len(nb_all)
    n_shadow = len(all_shadow)
    # Hybrid needs 50/50 split, so boundary half limits it to n_boundary
    # and random half needs n_boundary from shadow
    target_size = min(n_boundary, n_nb, n_shadow)
    # Hybrid needs even number (50/50 split)
    target_size = target_size - (target_size % 2)

    print(f"  Target training size: {target_size} (limited by smallest set)")

    # --- Boundary: trim to target ---
    boundary_data = boundary_data[:target_size]

    # --- Non-boundary: random sample to target ---
    idx_nb = np.random.choice(len(nb_all), target_size, replace=False)
    non_boundary_data = [nb_all[i] for i in idx_nb]

    # --- Hybrid: 50% boundary + 50% random from full shadow ---
    half = target_size // 2
    idx_hybrid_random = np.random.choice(len(all_shadow), half, replace=False)
    hybrid_data = boundary_data[:half] + [all_shadow[i] for i in idx_hybrid_random]

    # --- Random: from full shadow ---
    idx_random = np.random.choice(len(all_shadow), target_size, replace=False)
    random_data = [all_shadow[i] for i in idx_random]

    # Print distributions
    for label, group in [('Boundary', boundary_data),
                          ('Non-boundary', non_boundary_data),
                          ('Hybrid', hybrid_data),
                          ('Random', random_data)]:
        labels = [d.y.item() for d in group]
        unique, counts = np.unique(labels, return_counts=True)
        print(f"  {label}: {len(group)} samples, "
              f"class dist: {dict(zip(unique, counts))}")

    return boundary_data, non_boundary_data, hybrid_data, random_data

In [ ]:
# Run Unified Comparison
for name in ['AIDS', 'MUTAG', 'PTC_FM', 'NCI1', 'Tox21_AhR_training']:
    print(f"\n{'='*50}")
    print(f"Unified comparison: {name}")
    print(f"{'='*50}")

    ds = datasets[name]
    victim = victim_models[name]
    test_idx = dataset_splits[name]['test']
    shadow_idx = dataset_splits[name]['shadow']
    b = boundary_results[name]

    # Balance test set by victim predictions
    test_by_class = {}
    for idx in test_idx:
        pred = get_prediction(victim, ds[idx], device)
        test_by_class.setdefault(pred, []).append(idx)
    min_test = min(len(v) for v in test_by_class.values())
    balanced_test_idx = []
    for cls in sorted(test_by_class.keys()):
        indices = np.random.choice(test_by_class[cls], min_test, replace=False)
        balanced_test_idx.extend(indices)
    print(f"  Test: {len(balanced_test_idx)} balanced ({min_test}/class)")

    # Build all four training sets
    boundary_data, nb_data, hybrid_data, random_data = build_all_training_sets(
        ds, b['boundary_pairs'], b['non_boundary'], shadow_idx, victim, device
    )

    in_dim = get_feature_dim(ds)
    num_classes = ds.num_classes

    # Train and evaluate all four
    results = {}
    for strat_name, train_data in [('Boundary', boundary_data),
                                    ('Non-boundary', nb_data),
                                    ('Hybrid', hybrid_data),
                                    ('Random', random_data)]:
        print(f"  Training {strat_name} surrogate...")
        surr = train_surrogate(train_data, in_dim, num_classes, device)
        fid = evaluate_fidelity(surr, victim, ds, balanced_test_idx, device)
        results[strat_name] = fid

        # Prediction diagnostic
        surr.eval()
        preds = []
        with torch.no_grad():
            for idx in balanced_test_idx:
                data = ds[idx].clone().to(device)
                batch = torch.zeros(data.num_nodes, dtype=torch.long).to(device)
                preds.append(surr(data.x, data.edge_index, batch).argmax(1).item())
        s_unique, s_counts = np.unique(preds, return_counts=True)
        print(f"    Predictions: {dict(zip(s_unique, s_counts))}")

    print(f"\n  Results (balanced test):")
    print(f"    Boundary:     {results['Boundary']:.4f}")
    print(f"    Non-boundary: {results['Non-boundary']:.4f}")
    print(f"    Hybrid:       {results['Hybrid']:.4f}")
    print(f"    Random:       {results['Random']:.4f}")


Unified comparison: AIDS
  Test: 116 balanced (58/class)
  Target training size: 254 (limited by smallest set)
  Boundary: 254 samples, class dist: {np.int64(0): np.int64(127), np.int64(1): np.int64(127)}
  Non-boundary: 254 samples, class dist: {np.int64(0): np.int64(8), np.int64(1): np.int64(246)}
  Hybrid: 254 samples, class dist: {np.int64(0): np.int64(85), np.int64(1): np.int64(169)}
  Random: 254 samples, class dist: {np.int64(0): np.int64(26), np.int64(1): np.int64(228)}
  Training Boundary surrogate...
      Epoch 0: loss=22.2444
      Epoch 99: loss=14.8114
      Epoch 199: loss=13.3577
      Epoch 299: loss=10.3616
      Epoch 399: loss=9.2895
      Epoch 499: loss=9.9543
    Predictions: {np.int64(0): np.int64(62), np.int64(1): np.int64(54)}
  Training Non-boundary surrogate...
      Epoch 0: loss=12.1976
      Epoch 99: loss=1.0086
      Epoch 199: loss=0.0405
      Epoch 299: loss=3.6284
      Epoch 399: loss=0.0151
      Epoch 499: loss=0.0013
    Predictions: {np.int64(